In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🚀 API Versioning & Deprecation Strategy Guide

**Managing API evolution, backward compatibility, and graceful deprecation**

This guide covers:
- API versioning strategies (URL path, header, query parameter)
- Backward compatibility patterns
- Deprecation lifecycle and communication
- Version negotiation and content negotiation
- Breaking changes management
- API documentation for versions
- Migration paths for clients
- Sunset schedules and sunset headers
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. API Versioning Strategies

### URL Path Versioning
```python
from flask import Flask, jsonify, request

app = Flask(__name__)

@app.route('/api/v1/users/<user_id>', methods=['GET'])
def get_user_v1(user_id):
    """API v1 - returns user with basic info"""
    return jsonify({
        'id': user_id,
        'name': 'John Doe',
        'email': 'john@example.com'
    })

@app.route('/api/v2/users/<user_id>', methods=['GET'])
def get_user_v2(user_id):
    """API v2 - returns user with extended info"""
    return jsonify({
        'id': user_id,
        'name': 'John Doe',
        'email': 'john@example.com',
        'profile': {
            'bio': 'Software Engineer',
            'location': 'San Francisco',
            'social': ['github.com/john', 'twitter.com/john']
        },
        'created_at': '2023-01-15T10:30:00Z',
        'updated_at': '2024-01-20T14:45:00Z'
    })

# Route middleware for version support
@app.before_request
def log_api_version():
    """Log which API version is being used"""
    path = request.path
    if '/api/v' in path:
        version = path.split('/api/v')[1].split('/')[0]
        print(f"API Version: v{version}")
```

### Header-Based Versioning
```python
class VersionedAPI:
    """API with header-based versioning"""
    
    def get_user(self, user_id: str):
        """Get user with version-based response"""
        api_version = request.headers.get('API-Version', '1')
        
        if api_version == '1':
            return self._get_user_v1(user_id)
        elif api_version == '2':
            return self._get_user_v2(user_id)
        else:
            return {'error': f'Unsupported API version: {api_version}'}, 400
    
    def _get_user_v1(self, user_id: str):
        """Version 1 response format"""
        return {
            'id': user_id,
            'name': 'John Doe',
            'email': 'john@example.com'
        }
    
    def _get_user_v2(self, user_id: str):
        """Version 2 response format"""
        return {
            'id': user_id,
            'name': 'John Doe',
            'email': 'john@example.com',
            'profile': {
                'bio': 'Software Engineer',
                'location': 'San Francisco'
            }
        }
```

### Content Negotiation (Accept Header)
```python
from flask import make_response

class ContentNegotiationAPI:
    """API using content negotiation"""
    
    @staticmethod
    def get_data():
        """Return data based on Accept header"""
        accept = request.headers.get('Accept', 'application/json')
        
        data = {'id': 1, 'name': 'Example', 'value': 100}
        
        if 'application/json' in accept:
            return jsonify(data)
        elif 'application/xml' in accept:
            return ContentNegotiationAPI._to_xml(data), 200, {'Content-Type': 'application/xml'}
        elif 'text/csv' in accept:
            return ContentNegotiationAPI._to_csv(data), 200, {'Content-Type': 'text/csv'}
        else:
            return {'error': 'Unsupported media type'}, 406
    
    @staticmethod
    def _to_xml(data: dict) -> str:
        """Convert to XML"""
        import xml.etree.ElementTree as ET
        root = ET.Element('data')
        for key, value in data.items():
            ET.SubElement(root, key).text = str(value)
        return ET.tostring(root, encoding='unicode')
    
    @staticmethod
    def _to_csv(data: dict) -> str:
        """Convert to CSV"""
        import csv
        from io import StringIO
        output = StringIO()
        writer = csv.DictWriter(output, fieldnames=data.keys())
        writer.writeheader()
        writer.writerow(data)
        return output.getvalue()
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Deprecation Lifecycle

### Deprecation Manager
```python
from datetime import datetime, timedelta
from enum import Enum

class DeprecationStatus(Enum):
    ACTIVE = "active"
    DEPRECATED = "deprecated"
    SUNSET = "sunset"
    REMOVED = "removed"

class DeprecationPolicy:
    """Define deprecation timelines"""
    
    def __init__(
        self,
        endpoint: str,
        deprecation_date: datetime,
        sunset_date: datetime,
        replacement_endpoint: str = None
    ):
        self.endpoint = endpoint
        self.deprecation_date = deprecation_date
        self.sunset_date = sunset_date
        self.replacement_endpoint = replacement_endpoint
    
    def get_status(self) -> DeprecationStatus:
        """Get current deprecation status"""
        now = datetime.now()
        
        if now >= self.sunset_date:
            return DeprecationStatus.REMOVED
        elif now >= self.deprecation_date:
            return DeprecationStatus.DEPRECATED
        else:
            return DeprecationStatus.ACTIVE
    
    def get_days_until_sunset(self) -> int:
        """Days remaining until sunset"""
        remaining = (self.sunset_date - datetime.now()).days
        return max(0, remaining)
    
    def should_add_warning_header(self) -> bool:
        """Check if warning header should be added"""
        return self.get_status() in [DeprecationStatus.DEPRECATED, DeprecationStatus.SUNSET]

class APIDeprecationManager:
    """Manage deprecation across endpoints"""
    
    def __init__(self):
        self.policies = {}
    
    def register_policy(self, policy: DeprecationPolicy):
        """Register deprecation policy"""
        self.policies[policy.endpoint] = policy
    
    def get_deprecation_warning(self, endpoint: str) -> dict:
        """Get deprecation warning for endpoint"""
        policy = self.policies.get(endpoint)
        if not policy:
            return None
        
        status = policy.get_status()
        if status == DeprecationStatus.ACTIVE:
            return None
        
        return {
            'status': status.value,
            'replacement': policy.replacement_endpoint,
            'sunset_date': policy.sunset_date.isoformat(),
            'days_remaining': policy.get_days_until_sunset(),
            'message': f'This endpoint is {status.value} and will be removed on {policy.sunset_date.date()}'
        }

# Usage
manager = APIDeprecationManager()

# Register deprecation policy
policy = DeprecationPolicy(
    endpoint='/api/v1/users',
    deprecation_date=datetime.now(),
    sunset_date=datetime.now() + timedelta(days=90),
    replacement_endpoint='/api/v2/users'
)
manager.register_policy(policy)
```

### HTTP Sunset Header
```python
def add_deprecation_headers(endpoint: str):
    """Middleware to add deprecation headers"""
    
    def decorator(func):
        def wrapper(*args, **kwargs):
            response = func(*args, **kwargs)
            
            # Add deprecation notice header
            if endpoint in manager.policies:
                policy = manager.policies[endpoint]
                
                if policy.should_add_warning_header():
                    deprecation_warning = manager.get_deprecation_warning(endpoint)
                    
                    # RFC 8594 - Deprecation header
                    response.headers['Deprecation'] = 'true'
                    
                    # Sunset date (RFC 8230)
                    response.headers['Sunset'] = policy.sunset_date.strftime('%a, %d %b %Y %H:%M:%S GMT')
                    
                    # Link to replacement (RFC 8288)
                    if policy.replacement_endpoint:
                        response.headers['Link'] = f'<{policy.replacement_endpoint}>; rel="successor-version"'
                    
                    # Custom warning header
                    response.headers['Warning'] = f'299 - "{deprecation_warning["message"]}"'
            
            return response
        return wrapper
    return decorator

@add_deprecation_headers('/api/v1/users')
def get_users_v1():
    """Deprecated endpoint with warning headers"""
    return jsonify({'users': []})
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Breaking Changes Management

### Breaking Change Registry
```python
class BreakingChange:
    """Document a breaking change"""
    
    def __init__(
        self,
        version: str,
        endpoint: str,
        change_type: str,
        description: str,
        migration_guide: str,
        released_date: datetime
    ):
        self.version = version
        self.endpoint = endpoint
        self.change_type = change_type  # 'field_removed', 'field_renamed', 'response_format_changed'
        self.description = description
        self.migration_guide = migration_guide
        self.released_date = released_date

class BreakingChangeRegistry:
    """Registry of breaking changes"""
    
    def __init__(self):
        self.changes = []
    
    def register_change(self, change: BreakingChange):
        """Register a breaking change"""
        self.changes.append(change)
    
    def get_changelog(self) -> list:
        """Get changelog of breaking changes"""
        return sorted(
            self.changes,
            key=lambda x: x.released_date,
            reverse=True
        )
    
    def get_migration_path(self, from_version: str, to_version: str) -> list:
        """Get migration steps between versions"""
        changes = [c for c in self.changes 
                  if self._is_in_range(c.version, from_version, to_version)]
        return changes
    
    def _is_in_range(self, version: str, from_v: str, to_v: str) -> bool:
        """Check if version is in range"""
        # Simple version comparison (in production, use packaging.version)
        return int(version.split('.')[0]) >= int(from_v.split('.')[0]) and \
               int(version.split('.')[0]) <= int(to_v.split('.')[0])

# Usage
registry = BreakingChangeRegistry()

change = BreakingChange(
    version='2.0.0',
    endpoint='/api/users',
    change_type='field_removed',
    description='The "legacy_id" field has been removed',
    migration_guide='Update all references from user.legacy_id to user.id',
    released_date=datetime.now()
)
registry.register_change(change)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. API Versioning Checklist

✅ **Strategy**
- [ ] Versioning strategy chosen (URL path, header, content negotiation)
- [ ] Rationale documented
- [ ] Consistency across endpoints
- [ ] Client adoption tracked

✅ **Deprecation**
- [ ] Deprecation policy defined
- [ ] Timeline (6-12 months typical)
- [ ] Sunset headers implemented (RFC 8594)
- [ ] Communication plan executed
- [ ] Migration guide provided

✅ **Breaking Changes**
- [ ] All breaking changes documented
- [ ] Migration paths provided
- [ ] Backward compatibility layer (if possible)
- [ ] Client notification system
- [ ] Rollback plan available

✅ **Monitoring**
- [ ] Client adoption of new versions tracked
- [ ] Deprecated endpoint usage monitored
- [ ] Error rates by version tracked
- [ ] Migration success rate measured
- [ ] Support tickets for migration tracked
</VSCode.Cell>
```